# Commercial Labs Data Pipeline — Technical Walkthrough

**Repo:** [`amyshih-droid/commercial_labs`](https://github.com/amyshih-droid/commercial_labs)
**Prepared for:** mentor review
**Status:** Phase 2 (Pharma/Biotech) in progress · Phase 1 (Clinical/Diagnostic) on hold

This notebook is a **documentation and demo layer on top of the existing project** — it explains how
the pipeline works by reading and running the repo's real code, section by section. It does not
reimplement, rewrite, or improve any pipeline logic. Every function called below is imported directly
from the cloned repo; every code excerpt shown is read live from the cloned files, not retyped from
memory.

### How to read this notebook

Every code cell is labeled with one of these three tags, so it's always clear what kind of cell you're
looking at:

| Tag | Meaning |
|---|---|
| 📦 **PRODUCTION CODE** | The real source code, printed verbatim by reading the cloned file at runtime — this is exactly what's in the repo, not a paraphrase |
| 🔧 **DEMONSTRATION** | A small, fast example built by this notebook to illustrate how a production function behaves — not part of the actual pipeline invocation, and never used to regenerate the repo's real output files |
| 📊 **PRODUCTION OUTPUT** | Loading and inspecting a real result file that's already committed in the repo from a completed run — not regenerated here |

For stages that need paid API keys (OpenAI, billed Google Maps) or a local Chrome/Selenium install —
neither of which is available/appropriate to spend on a demo — the notebook reads the real source code
and the real committed output, without re-running the expensive step.


## 1. Architecture at a glance

**Goal (from `PROJECT_SPEC.md`):** build a clean, geocoded database of commercial wet labs (pharma/biotech
and environmental) in the US, for business-development prospecting.

**Data flow** (from `README.md`, and matches the stage order inside `scripts/run_pipeline.py`):

```
[ Connectors ]  9 pharma sources (APIs + scrapers) or 1 env source
      │
      ▼
[ Stage 1: standardize.py ]        raw columns  →  canonical schema (config/schema.yaml)
      │
      ▼
[ Stage 2: combine_standardized.py ]   stack all standardized/*.csv → auto_combined_raw.csv
      │
      ▼
[ Stage 3: entity_resolution.py ]  fuzzy name+city+state blocking → one row per real facility
      │
      ▼
[ Stage 4: llm_infer.py ]          async, page-targeted scraping + LLM extraction of missing fields
      │
      ▼
[ Stage 5: dedup_and_tag_entities.py ]  clean addresses, tag HQ vs. Branch, build display name
      │
      ▼
[ Stage 6: geocode_entities_google.py ] Google Maps Geocoding API → lat/lng
      │
      ▼
[ Stage 7: validate_before_db.py ] fill-rate + primary-key checks before DB load
```

**Two domains, one pipeline.** `run_pipeline.py --domain pharma` or `--domain env` runs the same 7 stages
against a different `DOMAIN_CONNECTORS` registry and a different `data/<domain>/` folder — none of the
stage functions themselves branch on domain.

**Config-driven, not code-driven.** `config/schema.yaml` defines the canonical schema every source must
map into; `config/source_mappings/<source>.yaml` declares how each source's raw columns map to it.
`standardize.py` itself has zero source-specific logic — this is verified directly in §5 below by reading
its actual source.


## 2. Setup

Clones the repo and installs the pinned dependency versions from the repo's own `requirements.txt`
(matching versions matters — a newer pandas changes groupby/apply semantics enough to break
`dedup_and_tag_entities.py` further down). Also defines `show_source()`, a small helper used throughout
this notebook to print real code straight from the cloned files — so every "production code" cell below
is guaranteed to be exactly what's in the repo, read at run time, not retyped.


In [ ]:
!git clone --quiet https://github.com/amyshih-droid/commercial_labs.git commercial_labs 2>/dev/null || echo "Repo already cloned, skipping."
%cd commercial_labs


In [ ]:
# Versions pinned to match requirements.txt exactly.
!pip install -q pandas==2.2.2 pyarrow>=16.0.0 pyyaml==6.0.2 rapidfuzz==3.9.6 requests==2.32.3 python-dotenv==1.0.1 plotly

import sys, os
sys.path.append("scripts")
sys.path.append("scripts/pipeline")

import pandas as pd
pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)

print("Working directory:", os.getcwd())
print("pandas version:", pd.__version__)


In [ ]:
def show_source(path, start=None, end=None):
    """Prints an excerpt of a real file from the cloned repo, read live from
    disk. `start`/`end` are 1-indexed, inclusive line numbers (like the repo
    viewer) - omit both to print the whole file. This is how every
    '📦 PRODUCTION CODE' cell in this notebook gets its content: nothing
    below is retyped or paraphrased, it's read directly from commercial_labs/.
    """
    with open(path) as f:
        lines = f.readlines()
    s = (start or 1) - 1
    e = end or len(lines)
    header = f"# ---- {path}"
    header += f" (lines {start}-{end}) ----" if start else " (full file) ----"
    print(header)
    print("".join(lines[s:e]))


## 3. `scripts/connectors/` — multi-source ingestion

**Purpose:** each connector's job is narrowly scoped to *fetching* one source's raw data into a
DataFrame — nothing else. That contract (`extract(headless=True) -> pd.DataFrame`, no saving, no
standardizing, no dedup, no LLM calls, no geocoding) is stated explicitly in each connector's own module
docstring, and is what lets `run_pipeline.py` loop over all 9 pharma connectors identically.

| Connector | Retrieval method | What it's for |
|---|---|---|
| `fda_hct.py` | Selenium (scrapes a dynamic ASP results table) | FDA Human Cell & Tissue establishment registrations |
| `fda_blood.py` | Selenium | FDA Blood Establishment Registration (BER) database |
| `fda_drug_registration.py` | Direct file download | FDA Drug Establishment Registration (largest single source) |
| `biopharmguy_therapeutics.py` | Selenium/scraping | Broad commercial biotech company directory |
| `biopharmguy_cmos.py` | Selenium/scraping | Contract manufacturers (CDMOs) |
| `biopharmguy_cros.py` | Selenium/scraping | Contract research orgs (CROs) |
| `form_d.py` | Bulk ZIP download + SEC API | SEC Form D — private capital raises (pre-FDA/pre-NIH stealth startups) |
| `ycombinator.py` | API | YC company directory, filtered to bio/therapeutics batches |
| `nih_reporter.py` | REST API | NIH SBIR/STTR grant recipients (early-stage R&D) |
| `env_labs.py` | Local file read (no connector config YAML) | TNI LAMS environmental lab export — the one source without an automated fetch, since it's a manual CSV download |

**Note on current state:** all 9 pharma connectors are commented out in `run_pipeline.py`'s
`DOMAIN_CONNECTORS` dict right now (shown in §13), so `--domain pharma` currently registers zero
connectors if invoked as-is. The `data/phase2_pharma/` folder still holds the full output of a real run
from when they *were* active — that's the production data inspected throughout this notebook. `env` is
the domain still wired up live.


### The connector contract, read from a Selenium-based connector

`fda_hct.py` needs a real Chrome browser via Selenium, so it isn't executed in this notebook — but its
docstring and public `extract()` function are worth reading, since they define the contract every
connector follows:


In [ ]:
show_source("scripts/connectors/fda_hct.py", 1, 17)

In [ ]:
show_source("scripts/connectors/fda_hct.py", 183, 201)

### `nih_reporter.py` — a connector that's safe to run live

This is the one pharma connector that's pure REST API (no browser, no scraping), so it's the one this
notebook actually executes. First, its core fetch logic and public API function, read from the repo:


In [ ]:
show_source("scripts/connectors/nih_reporter.py", 76, 162)

In [ ]:
show_source("scripts/connectors/nih_reporter.py", 193, 207)

**What this does:** `fetch_projects()` paginates through the NIH RePORTER v2 API (`offset`/`limit`),
filtering out academic/institutional recipients (`bad_keywords` — university, hospital, foundation, etc.)
since this project only wants commercial recipients, then flattens each project + its first principal
investigator into one flat dict. `extract()` wraps that in the standard connector contract: load
`config/connectors/nih_reporter.yaml` for the actual query parameters (activity codes, fiscal years,
record limits), call `fetch_projects()`, and validate the result has the required columns and a non-empty
primary key before returning it.

**Inputs:** nothing (reads its own config file). **Output:** a `pd.DataFrame` with columns like
`lab_name`, `street_address`, `city`, `state`, `contact_name`, `contact_email`, `grant_id`, etc.


🔧 **DEMONSTRATION** — a live call against the real NIH RePORTER API, using the unmodified connector code:

In [ ]:
from connectors import nih_reporter
import yaml

config = nih_reporter.load_config()
print("Real config/connectors/nih_reporter.yaml, as loaded by the connector:")
print(yaml.dump(config, sort_keys=False))


In [ ]:
# Live call - real API, real (unmodified) connector code, real config above
nih_df = nih_reporter.extract(headless=True)
print(f"Extracted {len(nih_df)} rows")
nih_df.head()


## 4. `config/` — the config-driven backbone

**Purpose:** two files here make `standardize.py` (Stage 1) source-agnostic:

- **`config/schema.yaml`** — the canonical master schema every source must be mapped into. Each field
  declares `required` (fail loudly if null after mapping) and, where relevant, `llm_fillable` +
  `llm_fill_method` (tells Stage 4 whether/how it may fill that field if a source doesn't provide it).
- **`config/source_mappings/<source>.yaml`** — one file per source: `direct_mappings` (raw column →
  canonical field), `composite_fields` (regex-based splitting of one combined column into several),
  row filters, and notes on which fields are intentionally left for LLM inference.

📦 **PRODUCTION CONFIG** — both files, read directly from the repo:


In [ ]:
show_source("config/schema.yaml")

In [ ]:
show_source("config/source_mappings/nih_reporter.yaml")

## 5. Stage 1 — `standardize.py`

**Purpose:** map any raw source's DataFrame into the canonical schema, using only the two YAML files
above — no source-specific branches in the Python at all.

📦 **PRODUCTION CODE** — `apply_direct_mappings()`, one of several mapping helpers `standardize_source()`
calls (renames + strips whitespace on every directly-mapped column):


In [ ]:
show_source("scripts/pipeline/standardize.py", 101, 122)

📦 **PRODUCTION CODE** — `standardize_source()`, the main entry point. Note it fills in *every* schema
field the source didn't produce (as null) before writing — this is what makes 9 differently-shaped
sources concatenable in Stage 2 without special-casing:


In [ ]:
show_source("scripts/pipeline/standardize.py", 367, 429)

**Inputs:** `source_key` (which `source_mappings/<key>.yaml` to use), `raw_file` (a parquet or CSV path),
`out_file` (where to write). **Output:** a CSV with exactly the columns in `config/schema.yaml`'s
`fields:` block, in that order, plus `source_record_id` and `source_name`.


🔧 **DEMONSTRATION** — running it, unmodified, on the NIH data fetched live above (mirrors exactly what `run_pipeline.py` does after each connector call):

In [ ]:
from standardize import standardize_source
from pathlib import Path
import datetime

demo_dir = Path("demo_run")
(demo_dir / "raw").mkdir(parents=True, exist_ok=True)
(demo_dir / "standardized").mkdir(parents=True, exist_ok=True)

raw_path = demo_dir / "raw" / f"nih_reporter_{datetime.date.today()}.parquet"
nih_df.to_parquet(raw_path, index=False)

std_path = demo_dir / "standardized" / "nih_reporter_standardized.csv"
standardize_source(
    source_key="nih_reporter",
    raw_file=raw_path,
    out_file=std_path,
    source_name_override="nih_reporter",
)

nih_standardized = pd.read_csv(std_path, dtype=str)
print(f"{len(nih_standardized)} rows, columns match config/schema.yaml exactly:")
list(nih_standardized.columns)


In [ ]:
nih_standardized.head(3)

## 6. Stage 2 — `combine_standardized.py`

**Purpose:** stack every standardized source on top of each other — no dedup, that's Stage 3's job. Just
concatenation with provenance (`source_file` column) so later stages have full traceability back to which
file a row came from.

📦 **PRODUCTION CODE** — the entire file is short enough to show in full:


In [ ]:
show_source("scripts/pipeline/combine_standardized.py")

**Inputs:** a directory of standardized CSVs. **Output:** one concatenated CSV with an added
`source_file` column. That's the whole stage.


🔧 **DEMONSTRATION** — rather than re-run this on the full 9-source production data (that's inspected
directly as real output in §8), here it is on a tiny 2-file input built from small real slices, so the
mechanism is visible in a couple of seconds:


In [ ]:
from combine_standardized import combine_csvs

# Small, real slices - first 3 rows of two of the repo's own standardized files
DATA_DIR = Path("data/phase2_pharma")
mini_dir = demo_dir / "mini_standardized"
mini_dir.mkdir(parents=True, exist_ok=True)

for fname in ["fda_hct_standardized.csv", "form_d_standardized.csv"]:
    pd.read_csv(DATA_DIR / "standardized" / fname, dtype=str).head(3).to_csv(mini_dir / fname, index=False)

combine_csvs(mini_dir, demo_dir / "mini_combined.csv")
pd.read_csv(demo_dir / "mini_combined.csv", dtype=str)[["company_name", "address_city", "address_state", "source_name", "source_file"]]


## 7. Stage 3 — `entity_resolution.py`

**Purpose:** collapse the combined multi-source table down to one row per real-world physical facility.
Deterministic and rule-based — no LLM involved in this stage.

📦 **PRODUCTION CODE** — normalization helpers, the blocking + fuzzy-match logic (`resolve_entities`),
and the clustering/merge logic (`run_entity_resolution`):


In [ ]:
show_source("scripts/pipeline/entity_resolution.py", 25, 126)

In [ ]:
show_source("scripts/pipeline/entity_resolution.py", 129, 191)

**Key design points, visible directly above:**
- Matches on **name + city + state together**, not name alone — same name in a different city is a
  different facility.
- A **blocking key** (`state | normalized_city | first 3 chars of normalized name`) limits comparisons to
  plausible pairs before any fuzzy matching runs — full pairwise comparison across the whole combined
  table would be far too slow. The code's own comment flags the tradeoff: a typo in a name's first 3
  characters could cause a missed match.
- `rapidfuzz.fuzz.token_sort_ratio` on the normalized name must clear `name_threshold` (default 90); if
  both rows have a street address, it must also clear `street_threshold` (default 75).
- A `UnionFind` (disjoint-set) structure clusters everything that matched, transitively, into one final
  entity per cluster. `merge_cluster()` then takes the first non-null value per field across each
  cluster's rows.

**Inputs:** the combined CSV from Stage 2, plus `name_threshold`/`street_threshold`. **Outputs:** two
files — `master_entities.csv` (one row per resolved entity, plus `entity_id`, `num_source_files`,
`source_files`) and `entity_crosswalk.csv` (traceability: which original source rows fed each entity).


🔧 **DEMONSTRATION** — a small, real slice built from `data/phase2_pharma/auto_combined_raw.csv` (the
repo's own committed Stage-2 output), filtered to 3 companies known to appear in more than one source, so
the merge behavior is actually visible on ~13 rows instead of the full ~16,000:


In [ ]:
combined_full = pd.read_csv(DATA_DIR / "auto_combined_raw.csv", dtype=str)

demo_slice = combined_full[
    combined_full["company_name"].str.upper().str.contains("AXOGEN|CELULARITY|FOCUS SCIENTIFIC", na=False)
].copy()
print(f"{len(demo_slice)} real rows selected across sources:")
demo_slice[["company_name", "address_city", "address_state", "source_name"]]


In [ ]:
from entity_resolution import run_entity_resolution

demo_slice.to_csv(demo_dir / "mini_combined_for_er.csv", index=False)
run_entity_resolution(
    input_file=demo_dir / "mini_combined_for_er.csv",
    out_entities=demo_dir / "mini_entities.csv",
    out_crosswalk=demo_dir / "mini_crosswalk.csv",
    name_threshold=90,
    street_threshold=75,
)


In [ ]:
mini_entities = pd.read_csv(demo_dir / "mini_entities.csv", dtype=str)
print(f"{len(demo_slice)} raw rows -> {len(mini_entities)} resolved entities\n")
mini_entities[["entity_id", "company_name", "address_city", "address_state", "num_source_files", "source_files"]]


Notice: 5 raw rows for "Celularity" at Florham Park, NJ — 1 from `fda_hct_standardized.csv` and 4 from
`form_d_standardized.csv` (repeat SEC filings under slightly different name spellings, `Celularity, Inc.`
vs `Celularity Inc`) — all collapse into one entity (`num_source_files: 2`, one per distinct source file).
Meanwhile the `CELULARITY` row at *Warren, NJ* stays a separate entity — correctly treated as a different
physical location, per the "name + city + state together" rule above.

For scale context (not re-run here — see §8's inspection of the real committed output): the full
production run collapses the actual ~16,364-row combined table into ~13,300 resolved entities.


## 8. Stage 4 — `llm_infer.py`

**Purpose:** fill in fields a source didn't provide (`address_street`, `contact_name`, `contact_email`,
`is_gmp_facility`, `is_commercial`, `services`) using a **website-first, page-targeted** strategy — never
a plain "ask the model what it knows" call.

📦 **PRODUCTION CODE** — the module's own docstring states the strategy and design principles precisely:


In [ ]:
show_source("scripts/pipeline/llm_infer.py", 1, 40)

📦 **PRODUCTION CODE** — the declarative routing tables the strategy above relies on: which page
categories are relevant to each missing field, and which link-text keywords identify each category on a
homepage's real navigation:


In [ ]:
show_source("scripts/pipeline/llm_infer.py", 79, 103)

📦 **PRODUCTION CODE** — the function `run_pipeline.py` actually calls (see §13):


In [ ]:
show_source("scripts/pipeline/llm_infer.py", 725, 744)

**Inputs:** the master entities CSV, a list of `fields` to fill, an output path, a cache directory.
**Output:** the same CSV with those fields filled where possible, plus a `<field>_status` column per
field recording *how* it was filled (`extracted`, `found_via_fallback_search`,
`checked_pages_not_found (...)`, etc.) — never silently null vs. filled.

**Why this isn't executed here:** it needs `OPENAI_API_KEY`, fetches real company websites, and the
repo's own `PROJECT_SPEC.md` records a full production run at **1h15m and ~12.7M tokens** across the
13,300-entity master table (~$0.80). That's a real cost/time tradeoff not worth spending on a demo, so
instead:


**A/B evaluation the author ran to validate this design** (from `PROJECT_SPEC.md` §5, Step 5) — a
baseline HTML-scraper-only version vs. the version described above (legal-name cleaning, suite-preserving
address prompts, grounded search fallback for contacts):

| Field | Baseline filled | Current filled | Net gain |
|---|---|---|---|
| `website_url` | 26/30 | 26/30 | +0 (fixed a false positive instead) |
| `address_street` | 27/30 | 27/30 | +0 (suite/floor granularity jumped 3.3% → 53.3%) |
| `contact_name` | 18/30 | 24/30 | **+6** |
| `contact_email` | 8/30 | 13/30 | **+5** |


📊 **PRODUCTION OUTPUT** — `auto_master_entities_enriched.csv`, the repo's real committed result of this stage on the full 13,300-entity master table:

In [ ]:
enriched = pd.read_csv(DATA_DIR / "auto_master_entities_enriched.csv", dtype=str)
print(f"{len(enriched)} entities, {len(enriched.columns)} columns")
enriched[["company_name", "website_url", "contact_name", "contact_email", "is_gmp_facility"]].sample(8, random_state=1)


In [ ]:
status_cols = [c for c in enriched.columns if c.endswith("_status")]
for c in status_cols:
    print(f"\n{c}:")
    simplified = enriched[c].dropna().str.replace(r"\s*\(.*\)", "", regex=True)
    print(simplified.value_counts().to_string())


## 9. Stage 5 — `dedup_and_tag_entities.py`

**Purpose:** clean up addresses, normalize company names for grouping, tag each location as HQ vs.
Branch, and build a human-readable composite `name`.

📦 **PRODUCTION CODE** — company-name normalization and street-address cleaning:


In [ ]:
show_source("scripts/pipeline/dedup_and_tag_entities.py", 28, 96)

📦 **PRODUCTION CODE** — HQ/Branch tagging, composite-name building, and the entry point that ties them
together:


In [ ]:
show_source("scripts/pipeline/dedup_and_tag_entities.py", 99, 229)

**Key design points:** `clean_and_validate_street()` drops a street value entirely if it contains a
different state's 2-letter code than the row's own `address_state` — a defense against hallucinated
cross-state addresses from Stage 4. `tag_hq_or_branch()` runs per company-name group: single-location
companies are HQ by default; multi-location groups look for an explicit HQ/headquarters keyword first,
then fall back to whichever row has a filled `cik` (SEC filers list their HQ address on Form D).
`build_composite_name()` only adds a street-disambiguating prefix when two locations would otherwise
collide on ZIP or city.

**Inputs:** the enriched CSV from Stage 4. **Output:** the same rows with `address_street` cleaned,
`company_name_clean`, `is_hq`, and `name` columns added.


🔧 **DEMONSTRATION** — running the unmodified function on a small real slice: every enriched row for
`TECOMET`, a company with multiple US locations in the real dataset (fast; the full 13,300-row run is
inspected as committed output right after):


In [ ]:
tecomet_slice = enriched[enriched["company_name"].str.upper() == "TECOMET"].copy()
tecomet_slice.to_csv(demo_dir / "tecomet_enriched.csv", index=False)

from dedup_and_tag_entities import run_dedup_and_tag
run_dedup_and_tag(input_file=demo_dir / "tecomet_enriched.csv", output_file=demo_dir / "tecomet_tagged.csv")

pd.read_csv(demo_dir / "tecomet_tagged.csv", dtype=str)[["name", "address_city", "address_state", "is_hq"]]


📊 **PRODUCTION OUTPUT** — the real committed result of this stage at full scale, `auto_master_entities_tagged.csv`:

In [ ]:
tagged_full = pd.read_csv(DATA_DIR / "auto_master_entities_tagged.csv", dtype=str)
hq_count = (tagged_full["is_hq"] == "True").sum()
print(f"{len(tagged_full)} total rows: {hq_count} tagged HQ, {len(tagged_full) - hq_count} tagged Branch")
tagged_full[tagged_full["company_name_clean"].isin(["TECOMET"])][["name", "address_city", "address_state", "is_hq"]]


## 10. Stage 6 — `geocode_entities_google.py`

**Purpose:** append latitude/longitude to every facility via the Google Maps Geocoding API.

📦 **PRODUCTION CODE** — the single-address lookup function and the batch-processing entry point:


In [ ]:
show_source("scripts/pipeline/geocode_entities_google.py", 56, 111)

In [ ]:
show_source("scripts/pipeline/geocode_entities_google.py", 113, 215)

**Key design points:** results are cached on disk (`geocode_cache_google.json`) keyed by a normalized
address string, so re-running the pipeline never re-bills an address it's already resolved.
`OVER_QUERY_LIMIT` responses are retried with backoff; `REQUEST_DENIED`/`INVALID_REQUEST` fail fast
without retrying, since retrying can't fix a bad key or malformed query. Google's own `location_type`
(`ROOFTOP`, `RANGE_INTERPOLATED`, `GEOMETRIC_CENTER`, `APPROXIMATE`) is preserved as
`google_geocode_source` so downstream consumers can judge match precision, not just whether a match
exists.

**Inputs:** the tagged CSV from Stage 5, `only_missing` (skip rows already geocoded). **Output:** the same
rows with `google_latitude`, `google_longitude`, `google_geocode_source`, and
`google_geocode_formatted_address` added.

**Why this isn't executed at scale here:** it needs a billed `GOOGLE_MAPS_API_KEY`. If you have your own
key, the cell below will run a real 5-row smoke test live; otherwise it loads the real committed output.


In [ ]:
import os

RUN_LIVE_GEOCODING = bool(os.environ.get("GOOGLE_MAPS_API_KEY"))

if RUN_LIVE_GEOCODING:
    print("GOOGLE_MAPS_API_KEY found - running a small live smoke test (5 rows, billed).")
    from geocode_entities_google import run_geocode_google
    run_geocode_google(
        input_file=DATA_DIR / "auto_master_entities_tagged.csv",
        output_file=demo_dir / "mini_geocoded.csv",
        only_missing=True,
        limit=5,
    )
    geocoded_demo = pd.read_csv(demo_dir / "mini_geocoded.csv", dtype=str)
    print(geocoded_demo[["name", "google_latitude", "google_longitude", "google_geocode_source"]].head())
else:
    print("No GOOGLE_MAPS_API_KEY set - skipping the live (billed) call.")


📊 **PRODUCTION OUTPUT** — `auto_master_entities_geocoded.csv`, the repo's real committed final geocoded dataset:

In [ ]:
geocoded = pd.read_csv(DATA_DIR / "auto_master_entities_geocoded.csv", dtype=str)
print(f"{geocoded['google_latitude'].notna().sum()}/{len(geocoded)} rows have coordinates")
geocoded[["name", "address_city", "address_state", "google_latitude", "google_longitude", "google_geocode_source"]].head()


A visual for the mentor, built from the real committed coordinates:

In [ ]:
import plotly.express as px

map_df = geocoded.dropna(subset=["google_latitude", "google_longitude"]).copy()
map_df["google_latitude"] = map_df["google_latitude"].astype(float)
map_df["google_longitude"] = map_df["google_longitude"].astype(float)

fig = px.scatter_geo(
    map_df,
    lat="google_latitude",
    lon="google_longitude",
    hover_name="name",
    color="source_name" if "source_name" in map_df.columns else None,
    scope="usa",
    title=f"Commercial labs - pharma/biotech domain ({len(map_df)} geocoded facilities, real committed data)",
    opacity=0.6,
)
fig.update_layout(height=550, margin=dict(l=0, r=0, t=40, b=0))
fig.show()


## 11. Stage 7 — `validate_before_db.py`

**Purpose:** the final quality gate before anything is considered ready for DB ingestion.

📦 **PRODUCTION CODE** — `validate_dataset()`, the whole check:


In [ ]:
show_source("scripts/pipeline/validate_before_db.py", 10, 113)

**What it checks, in order:** (1) required base fields (`company_name`, `entity_id`) present and
non-null — fatal if not; (2) no duplicate `source_record_id` values — fatal if not; (3) fill rate per
business-critical field; (4) a breakdown of every `*_status` column across the whole dataset.

**Inputs:** a file path. **Output:** log lines only (no file written) — `sys.exit(1)` on a fatal check,
otherwise a pass-through.

Pure pandas, cheap even at full scale (13,300 rows in a few seconds) — so unlike Stages 4/6, this one is
run for real, live, on the actual final committed file:


In [ ]:
from validate_before_db import validate_dataset

# validate_dataset() calls sys.exit(1) on a fatal check - catch that so this
# cell reports the real result instead of just halting.
try:
    validate_dataset(DATA_DIR / "auto_master_entities_geocoded.csv")
except SystemExit as e:
    print(f"\n[validate_before_db exited with code {e.code} - see the CRITICAL log line(s) above]")


**Running this on the real committed final output surfaces a genuine, current data-quality issue:** the
gate reports **277 duplicate `source_record_id` values** and exits with a failure — this exact committed
file would not pass its own pre-DB gate as-is. Spot-checking one duplicated ID shows it's shared by two
*different* `entity_id`s (the same underlying source record ended up in two separate resolved entities),
which points at Stage 3's blocking key being slightly too strict in at least a few cases — the code's own
comment on that tradeoff is quoted in §7 above. Flagging this as a real finding worth a closer look, not
fixing it here — this notebook only reads and demonstrates the existing pipeline.


## 12. `run_pipeline.py` — how it all ties together

**Purpose:** the single orchestrator that runs all 7 stages in order for one domain.

📦 **PRODUCTION CODE** — the entire file:


In [ ]:
show_source("scripts/run_pipeline.py")

**Design points worth calling out, visible directly above:**
- Every stage's `try/except` block only logs (`logger.error(..., exc_info=True)`) rather than raising —
  one connector failing (e.g. a scraper breaking when a site changes its HTML) doesn't take down the rest
  of the run.
- Every stage function is imported and called as a **native Python function**, not shelled out as a
  subprocess — exceptions propagate normally and there's no subprocess/stdout-parsing overhead between
  stages.
- The pharma/env split is entirely a different `DOMAIN_CONNECTORS` entry and a different `data/<domain>/`
  path — none of the 7 imported stage functions branch on domain internally.
- The 9 pharma connectors are currently commented out inside `DOMAIN_CONNECTORS`, confirming what §3
  described.

**Command to run this for real** (from the README, not executed in this notebook):
```bash
python3 scripts/run_pipeline.py --domain env      # currently wired up
python3 scripts/run_pipeline.py --domain pharma   # connectors currently commented out
```


## 13. `tests/` — current test coverage

Two test files exist in the repo. Rather than describe them, this section runs them for real:


In [ ]:
show_source("tests/test_geocode_google.py", 16, 22)

The `MODULE_PATH` shown above is worth remembering for what comes next. Running the real test suite:

In [ ]:
!pip install -q pytest
!python -m pytest tests/ -v --no-header 2>&1 | tail -n 40


**What this shows, running the real, unmodified test files:**
- `tests/test_fda_hct.py` is an **empty file** (0 bytes) — a placeholder that was never filled in.
- `tests/test_geocode_google.py` loads its target module by file path (`MODULE_PATH` above), rather than
  a normal `import`, because the original filename started with a digit
  (`07b_geocode_entities_google.py`) — not a valid Python module name. At some point the pipeline scripts
  were renamed to drop their numeric prefixes (it's now `scripts/pipeline/geocode_entities_google.py`),
  but this test's hardcoded `MODULE_PATH` still points at the old `07b_...` filename, so it fails to
  import its target and every test in the file errors out. The 6 test cases themselves (success,
  zero-results, rate-limit retry/backoff, request-denied fast-fail, empty-address short-circuit, network-
  exception handling) look well-designed — this looks like a one-line fix (`MODULE_PATH`) that just didn't
  get made when the scripts were renamed. Flagging it, not fixing it, per this notebook's scope.


## 14. Project spec & roadmap — key decisions

Summarized from `PROJECT_SPEC.md` and `ROADMAP.md` (both version-controlled in the repo root):

**Scope & phasing**
- **Phase 1 (clinical/diagnostic):** CMS CLIA database, California → all 50 states. **On hold**, not
  current priority.
- **Phase 2 (pharma/biotech):** no single free registry exists, so data is sourced across 9 connectors.
  **In progress** — this is the domain this notebook covers.

**Notable resolved decisions** (from the spec's decision log):
- Null convention: empty string `""` everywhere, never `NA`/`null`/`None` — avoids mixed-convention bugs
  across 9 differently-formatted sources.
- HQ vs. branch locations are **not** duplicates and both get kept; true duplicates are only collapsed
  when they represent the same physical facility (this is exactly Stage 3's logic in §7 above).
- A "commercial lab" explicitly **excludes** government labs, university/academic research labs, internal
  hospital labs, and non-profit health systems — the spec calls out that CMS's own control-type field
  doesn't perfectly capture this distinction, as a known data-quality limitation rather than an assumed-
  clean input.

**Roadmap — open items** (from `ROADMAP.md`, as of the last update):
- [ ] Fix GMP fallback overwrites in Stage 4 (a Google fallback search can currently trigger even when
      the initial scrape found `"no_explicit_gmp"`, which should count as "checked, found nothing" rather
      than "missing")
- [ ] Universities and public hospitals should not be included in the commercial-labs output — filtering
      gap still being tightened


## 15. Summary

| Question | Answer |
|---|---|
| How many pharma entities in the current production dataset? | ~13,300 resolved entities across 9 sources (`auto_master_entities.csv`) |
| How many are geocoded in the committed output? | 13,300/13,300 (100%) in `auto_master_entities_geocoded.csv` |
| What's the biggest single source? | FDA Drug Establishment Registration (DRLS) |
| Does the final committed output pass its own validation gate? | **No** — 277 duplicate `source_record_id`s, see §11 |
| What needs credentials to reproduce at full scale? | Stage 4 (`OPENAI_API_KEY`) and Stage 6 (billed `GOOGLE_MAPS_API_KEY`); the Selenium-based connectors additionally need a local Chrome install |
| What's demonstrated live in this notebook? | Stages 1-3, 5, 7 on real (small or full) data; Stage 4 and 6's real committed output is inspected without re-running them |

**For a live walkthrough with your mentor:** the connector demo (§3), the small entity-resolution merge
example (§7), and the map (§10) are the most visually convincing sections to run interactively; §8, §12,
and §14 are the best sections to talk through rather than execute.
